# Predictive Maintenance — Data Preprocessing

This notebook prepares the AI4I 2020 Predictive Maintenance Dataset for machine learning.

## Objectives

- Verify the dataset structure and data types
- Remove irrelevant identifier columns
- Remove failure-indicator columns that may cause target leakage
- Separate features and target
- Split the dataset into training and testing sets
- Encode categorical variables
- Scale numerical variables
- Build a reproducible preprocessing pipeline

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/raw/ai4i2020.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
print("Dataset shape:", df.shape)

Dataset shape: (10000, 14)


In [5]:
print(df.dtypes)

UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object


In [6]:
data = df.copy()

In [7]:
print("Original shape:", df.shape)
print("Working shape:", data.shape)

Original shape: (10000, 14)
Working shape: (10000, 14)


In [8]:
missing_values = data.isnull().sum()

print(missing_values)

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64


In [9]:
print("Total missing values:", data.isnull().sum().sum())

Total missing values: 0


In [10]:
duplicate_count = data.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [11]:
identifier_columns = [
    "UDI",
    "Product ID"
]

In [12]:
data = data.drop(columns=identifier_columns)

In [13]:
print(data.columns.tolist())

['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


In [14]:
leakage_columns = [
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

In [15]:
data = data.drop(columns=leakage_columns)

In [16]:
print(data.columns.tolist())

['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']


In [17]:
print("Cleaned dataset shape:", data.shape)

Cleaned dataset shape: (10000, 7)


In [18]:
data.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [19]:
feature_columns = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

In [20]:
target_column = "Machine failure"

In [21]:
X = data[feature_columns]
y = data[target_column]

In [22]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 6)
y shape: (10000,)


In [23]:
print(X.dtypes)

Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
dtype: object


In [24]:
categorical_features = [
    "Type"
]

numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

In [25]:
from sklearn.model_selection import train_test_split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [27]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 6)
X_test: (2000, 6)
y_train: (8000,)
y_test: (2000,)


In [28]:
print("Training distribution:")
print(y_train.value_counts())

print("\nTraining percentage:")
print(y_train.value_counts(normalize=True) * 100)

Training distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Training percentage:
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64


In [29]:
print("\nTesting distribution:")
print(y_test.value_counts())

print("\nTesting percentage:")
print(y_test.value_counts(normalize=True) * 100)


Testing distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64

Testing percentage:
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


In [30]:
stratify=y

In [33]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [35]:
X_train_processed = preprocessor.fit_transform(X_train)

In [36]:
X_test_processed = preprocessor.transform(X_test)

In [38]:
print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)

print("\nOriginal X_test shape:", X_test.shape)
print("Processed X_test shape:", X_test_processed.shape)

Original X_train shape: (8000, 6)
Processed X_train shape: (8000, 8)

Original X_test shape: (2000, 6)
Processed X_test shape: (2000, 8)


In [39]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['num__Air temperature [K]' 'num__Process temperature [K]'
 'num__Rotational speed [rpm]' 'num__Torque [Nm]' 'num__Tool wear [min]'
 'cat__Type_H' 'cat__Type_L' 'cat__Type_M']


In [40]:
processed_path = "../data/processed/cleaned_data.csv"

data.to_csv(processed_path, index=False)

print(f"Cleaned dataset saved to: {processed_path}")

Cleaned dataset saved to: ../data/processed/cleaned_data.csv


In [41]:
print("=" * 50)
print("FINAL PREPROCESSING CHECK")
print("=" * 50)

print("\nOriginal dataset:")
print(df.shape)

print("\nCleaned dataset:")
print(data.shape)

print("\nFeatures:")
print(feature_columns)

print("\nTarget:")
print(target_column)

print("\nTraining data:")
print(X_train.shape)

print("\nTesting data:")
print(X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nMissing values:")
print(data.isnull().sum().sum())

print("\nDuplicate rows:")
print(data.duplicated().sum())

print("\nProcessed training shape:")
print(X_train_processed.shape)

print("\nProcessed testing shape:")
print(X_test_processed.shape)

FINAL PREPROCESSING CHECK

Original dataset:
(10000, 14)

Cleaned dataset:
(10000, 7)

Features:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

Target:
Machine failure

Training data:
(8000, 6)

Testing data:
(2000, 6)

Training target distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing target distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64

Missing values:
0

Duplicate rows:
0

Processed training shape:
(8000, 8)

Processed testing shape:
(2000, 8)


## Preprocessing Findings

1. The original dataset contains 10,000 observations and 14 columns.
2. The dataset contains no missing values and no duplicate rows, so no imputation or duplicate removal is required.
3. `UDI` and `Product ID` are identifier columns and are excluded from the machine learning feature set.
4. `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are excluded from the initial predictive model because they represent specific failure indicators and may introduce target leakage.
5. The final input feature set consists of `Type`, air temperature, process temperature, rotational speed, torque, and tool wear.
6. `Machine failure` is used as the binary target variable.
7. `Type` is a categorical feature and will be handled using One-Hot Encoding.
8. The five continuous/numerical machine-condition features are processed using `StandardScaler`.
9. The dataset is divided into 80% training and 20% testing data using stratified sampling to preserve the minority failure class distribution.
10. The preprocessing transformer is fitted only on the training data and then applied to the test data to prevent data leakage.
11. After preprocessing, the six original input features are represented by eight model-ready features because the three categories of `Type` are one-hot encoded.
12. The resulting training and testing data are ready for exploratory analysis and subsequent machine learning experiments.